# Notebook 01 — Data Loading

**Goal:** Load the two CSV datasets, inspect structure, document column types and null counts.

**Datasets:**
- `data/raw/marathon/` — Marathon Results 2000–2019 (Zenodo 6959864)
- `data/raw/athletes/` — Long-Distance Running biometrics (Kaggle: mexwell)

Nothing is modified here — this notebook is read-only inspection.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

RAW_DIR = '../data/raw'
MARATHON_DIR = os.path.join(RAW_DIR, 'marathon')
ATHLETES_DIR = os.path.join(RAW_DIR, 'athletes')

print('Marathon files found:', glob.glob(os.path.join(MARATHON_DIR, '*.csv')))
print('Athlete files found: ', glob.glob(os.path.join(ATHLETES_DIR, '*.csv')))

## 1. Marathon Results Dataset

In [ ]:
# Load all marathon CSVs from the Zenodo record and concatenate them.
marathon_files = glob.glob(os.path.join(MARATHON_DIR, '*.csv'))

if not marathon_files:
    raise FileNotFoundError(
        'No marathon CSVs found. Download from https://zenodo.org/record/6959864 '
        'and place files in data/raw/marathon/'
    )

frames = []
for f in sorted(marathon_files):
    df_part = pd.read_csv(f, low_memory=False)
    df_part['source_file'] = os.path.basename(f)
    frames.append(df_part)
    print(f'Loaded {os.path.basename(f):40s} — {len(df_part):>8,} rows, {df_part.shape[1]} cols')

marathon_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal marathon rows: {len(marathon_raw):,}')

In [ ]:
print('=== Marathon columns ===')
print(marathon_raw.dtypes)
print()
marathon_raw.head(3)

In [ ]:
print('=== Null counts ===')
nulls = marathon_raw.isnull().sum()
print(nulls[nulls > 0])
print(f'\nNull percentage per column:')
print((nulls[nulls > 0] / len(marathon_raw) * 100).round(2))

In [ ]:
print('=== Summary statistics ===')
marathon_raw.describe(include='all')

In [ ]:
# Inspect the schema: print unique values for categorical columns.
for col in marathon_raw.columns:
    n_unique = marathon_raw[col].nunique()
    if n_unique < 30:
        print(f'{col} ({n_unique} unique): {sorted(marathon_raw[col].dropna().unique().tolist())}')
    else:
        sample = marathon_raw[col].dropna().head(3).tolist()
        print(f'{col} ({n_unique} unique): sample = {sample}')

In [ ]:
# Locate the time / age / gender columns by keyword.
time_candidates = [c for c in marathon_raw.columns if any(k in c.lower() for k in ['time', 'finish', 'official'])]
age_candidates  = [c for c in marathon_raw.columns if 'age' in c.lower()]
gender_candidates = [c for c in marathon_raw.columns if any(k in c.lower() for k in ['gender', 'sex', 'm/f'])]

print('Finish time candidates:', time_candidates)
print('Age candidates:        ', age_candidates)
print('Gender candidates:     ', gender_candidates)

## 2. Athlete Biometrics Dataset

In [ ]:
athlete_files = glob.glob(os.path.join(ATHLETES_DIR, '*.csv'))

if not athlete_files:
    raise FileNotFoundError(
        'No athlete CSVs found. Download from '
        'https://kaggle.com/datasets/mexwell/long-distance-running-dataset '
        'and place files in data/raw/athletes/'
    )

athletes_raw = pd.concat(
    [pd.read_csv(f) for f in athlete_files],
    ignore_index=True
)
print(f'Athlete rows: {len(athletes_raw):,}')
athletes_raw.head(3)

In [ ]:
print('=== Athlete columns ===')
print(athletes_raw.dtypes)
print()
print('=== Null counts ===')
nulls = athletes_raw.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls')

In [ ]:
athletes_raw.describe()